<a href="https://colab.research.google.com/github/williamscarson093-cloud/steam-smart-buy/blob/main/steam_smart_buy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Smart Buy

My app helps money conscious gamers decide which Steam game to buy by comparing games that fit their budget and genre, checking current deals, and weighing review.

In [34]:

import pandas as pd
import requests

url = "https://raw.githubusercontent.com/williamscarson093-cloud/steam-smart-buy/main/steam_games.csv"

df = pd.read_csv(url)

print(f"Successfully loaded {len(df)} games!")


Successfully loaded 1000 games!


In [35]:
print(df.columns.tolist())

['appid', 'name', 'release_date', 'release_year', 'price', 'discount', 'required_age', 'windows', 'mac', 'linux', 'developers', 'publishers', 'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive', 'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners', 'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description', 'critic_reviews']


In [36]:
def popular_games(games):
    popular = games[games["num_reviews_total"] >= 10000]
    return popular[["name", "num_reviews_total"]]

print(popular_games(df).head(5))

                              name  num_reviews_total
0                 Counter-Strike 2            8632939
1              PUBG: BATTLEGROUNDS            2513842
2                           Dota 2            2452595
3        Grand Theft Auto V Legacy            1803832
4  Tom Clancy's Rainbow Six® Siege            1168020


In [37]:
def find_games(games, genre, budget):
    matches = games[
        (games["genres"].str.contains(genre, case=False, na=False, regex=False)) &
        (games["price"] <= budget)
    ]

    matches = matches.sort_values("pct_pos_total", ascending=False)

    return matches[["name", "genres", "price", "pct_pos_total", "num_reviews_total"]]

results = find_games(df, "Action", 20)
print(results.head(10))

                          name  \
473  HoloCure - Save the Fans!   
63           Vampire Survivors   
311                Katana ZERO   
320                Pizza Tower   
54           People Playground   
6                     Terraria   
878            The Expendabros   
711                Wobbly Life   
605          The Wolf Among Us   
125     Mount & Blade: Warband   

                                                genres  price  pct_pos_total  \
473                   Action, Adventure, Casual, Indie   0.00             99   
63                          Action, Casual, Indie, RPG   4.99             98   
311                                      Action, Indie  14.99             98   
320                                      Action, Indie  19.99             98   
54                   Action, Casual, Indie, Simulation   9.99             98   
6                        Action, Adventure, Indie, RPG   9.99             97   
878             Action, Adventure, Indie, Free To Play   0.00    

In [45]:

import requests

def check_deals(game_name):
    url = "https://www.cheapshark.com/api/1.0/games"

    params = {"title": game_name}

    headers = {
        "User-Agent": "SteamSmartBuy/1.0 (student project)"
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()
        return response.json()

    except requests.RequestException:
        print("Could not check deals for:", game_name)
        return []


In [39]:

def recommend_games(games, genre, budget):
    matches = find_games(games, genre, budget)

    if matches.empty:
        print("No games found.")
        return matches

    print("Top games for your budget:")
    print(matches.head(5))

    return matches.head(5)

recommendations = recommend_games(df, "Action", 20)


Top games for your budget:
                          name                             genres  price  \
473  HoloCure - Save the Fans!   Action, Adventure, Casual, Indie   0.00   
63           Vampire Survivors         Action, Casual, Indie, RPG   4.99   
311                Katana ZERO                      Action, Indie  14.99   
320                Pizza Tower                      Action, Indie  19.99   
54           People Playground  Action, Casual, Indie, Simulation   9.99   

     pct_pos_total  num_reviews_total  
473             99              37057  
63              98             234756  
311             98              61243  
320             98              59430  
54              98             258453  


In [40]:

class SteamSmartBuy:
    def __init__(self, games):
        self.games = games

    def recommend(self, genre, budget):
        return recommend_games(self.games, genre, budget)

    def deals(self, game_name):
        return check_deals(game_name)


app = SteamSmartBuy(df)

app.recommend("Action", 20)
app.deals("Terraria")


Top games for your budget:
                          name                             genres  price  \
473  HoloCure - Save the Fans!   Action, Adventure, Casual, Indie   0.00   
63           Vampire Survivors         Action, Casual, Indie, RPG   4.99   
311                Katana ZERO                      Action, Indie  14.99   
320                Pizza Tower                      Action, Indie  19.99   
54           People Playground  Action, Casual, Indie, Simulation   9.99   

     pct_pos_total  num_reviews_total  
473             99              37057  
63              98             234756  
311             98              61243  
320             98              59430  
54              98             258453  


[{'gameID': '39',
  'steamAppID': '105600',
  'cheapest': '9.99',
  'cheapestDealID': '%2B7y%2FjZTRXxyTajQx%2FtvBN1%2BoisI6Iv5D7TL9ma6o7lU%3D',
  'external': 'Terraria',
  'internalName': 'TERRARIA',
  'thumb': 'https://shared.fastly.steamstatic.com/store_item_assets/steam/apps/105600/capsule_231x87.jpg?t=1792530636'}]

In [41]:

def compare_deals(games, genre, budget):
    matches = find_games(games, genre, budget)
    matches = matches[matches["num_reviews_total"] >= 10000]

    print("Checking deals for recommended games:")

    for name in matches["name"].head(5):
        print("\n", name)
        check_deals(name)

    return matches.head(5)

final_games = compare_deals(df, "Action", 20)


Checking deals for recommended games:

 HoloCure - Save the Fans!

 Vampire Survivors

 Katana ZERO

 Pizza Tower

 People Playground


In [42]:

def compare_deals(games, genre, budget):
    matches = find_games(games, genre, budget)
    matches = matches[matches["num_reviews_total"] >= 10000]

    print("Checking deals for recommended games:\n")

    results = []

    for name in matches["name"].head(5):
        deals = check_deals(name)

        # Only keep exact game matches, not DLC
        exact = [
            deal for deal in deals
            if deal["external"].lower() == name.lower()
        ]

        if exact:
            cheapest = min(
                exact,
                key=lambda deal: float(deal["cheapest"])
            )

            results.append({
                "Game": name,
                "Lowest Price": float(cheapest["cheapest"])
            })

    return pd.DataFrame(results)


final_games = compare_deals(df, "Action", 20)
print(final_games)


Checking deals for recommended games:

                Game  Lowest Price
0  Vampire Survivors          4.99
1        Katana ZERO          8.99
2        Pizza Tower         19.99
3  People Playground          9.99


In [43]:

def show_discounts(games):
    results = games.copy()

    # Match each game with its dataset price
    results = results.merge(
        df[["name", "price"]],
        left_on="Game",
        right_on="name",
        how="left"
    )

    results["Savings"] = (
        results["price"] - results["Lowest Price"]
    ).clip(lower=0).round(2)

    results["Discount %"] = (
        results["Savings"] /
        results["price"].replace(0, float("nan")) * 100
    ).fillna(0).round(1)

    results = results.sort_values(
        "Discount %", ascending=False
    )

    return results[
        ["Game", "price", "Lowest Price",
         "Savings", "Discount %"]
    ]


discount_results = show_discounts(final_games)
print(discount_results.to_string(index=False))


             Game  price  Lowest Price  Savings  Discount %
      Katana ZERO  14.99          8.99      6.0        40.0
Vampire Survivors   4.99          4.99      0.0         0.0
      Pizza Tower  19.99         19.99      0.0         0.0
People Playground   9.99          9.99      0.0         0.0


In [44]:

from IPython.display import display
import ipywidgets as widgets

genre_input = widgets.Dropdown(
    options=["Action", "Adventure", "RPG",
             "Strategy", "Simulation", "Indie"],
    value="Action",
    description="Genre:"
)

budget_input = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=5,
    description="Budget: $"
)

search_button = widgets.Button(
    description="Find My Games",
    button_style="success"
)

output = widgets.Output()

def search_games(button):
    with output:
        output.clear_output()

        print("STEAM SMART BUY")
        print("--------------------")

        games_found = compare_deals(
            df,
            genre_input.value,
            budget_input.value
        )

        if games_found.empty:
            print("No matching games found.")
        else:
            results = show_discounts(games_found)
            display(results)

search_button.on_click(search_games)

display(genre_input, budget_input, search_button, output)


STEAM SMART BUY
--------------------
Checking deals for recommended games:



,Game,price,Lowest Price,Savings,Discount %
1,Katana ZERO,14.99,8.99,6.0,40.0
0,Vampire Survivors,4.99,4.99,0.0,0.0
2,Pizza Tower,19.99,19.99,0.0,0.0
3,People Playground,9.99,9.99,0.0,0.0
